# Stage 3: Prepare WRF Climate Change Scenarios

**Purpose:** Filter urbanisation scenario DataFrames (discharge only, no rainfall
columns) and compute event-level statistics, ready for the WRF climate analysis.

This notebook replaces the two hardcoded legacy scripts:
- `scripts/run_filter_38.py`  (`38% urbanization`)
- `scripts/run_filter_87.py`  (`87% urbanization`)

## Scope
This notebook handles **data export only**.  WRF hydrological analysis, plotting,
and historical-vs-future comparisons will be implemented in Part 2 of this project.

## Outputs
All files are written to `outputs/models/scenarios/` — legacy directories are untouched.


In [1]:
# ── Configuration ────────────────────────────────────────────────────────────
import sys
from pathlib import Path

sys.path.insert(0, str(Path('.').resolve()))

# Source data (read-only)
SIM_DIR = Path(
    r'D:\Development\RESEARCH\Raanana\SWMM\from_radar'
    r'\Climate_Change\pickles\Urbanization_comparsion'
)

# All outputs go here
PROJECT_ROOT = Path('.')
SCENARIOS_DIR = PROJECT_ROOT / 'outputs' / 'models' / 'scenarios'
SCENARIOS_DIR.mkdir(parents=True, exist_ok=True)

# Select which scenarios to export.
# Use None to export all 10 scenarios in PKL_MAP.
# Use a list of keys to export a subset (e.g. the two legacy scripts' targets).
EXPORT_SCENARIOS = [
    '38% urbanization',   # replaces run_filter_38.py
    '87% urbanization',   # replaces run_filter_87.py
]

print(f'Source dir    : {SIM_DIR}')
print(f'Output dir    : {SCENARIOS_DIR}')
print(f'Scenarios     : {EXPORT_SCENARIOS}')


Source dir    : D:\Development\RESEARCH\Raanana\SWMM\from_radar\Climate_Change\pickles\Urbanization_comparsion
Output dir    : outputs\models\scenarios
Scenarios     : ['38% urbanization', '87% urbanization']


In [2]:
# ── Imports ───────────────────────────────────────────────────────────────────
import logging
import pandas as pd

from urban_runoff.scenarios.export import (
    PKL_MAP,
    export_scenario,
    export_all_scenarios,
)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)-8s %(name)s: %(message)s',
    datefmt='%H:%M:%S',
)
logger = logging.getLogger('scenarios_notebook')

print('Available scenarios:')
for key, fname in PKL_MAP.items():
    print(f'  {key:<20} -> {fname}')


Available scenarios:
  38% urbanization     -> urbanization_1_0.pkl
  42% urbanization     -> urbanization_1_1.pkl
  46% urbanization     -> urbanization_1_2.pkl
  49% urbanization     -> urbanization_1_3.pkl
  53% urbanization     -> urbanization_1_4.pkl
  57% urbanization     -> urbanization_1_5.pkl
  73% urbanization     -> urbanization_2_0.pkl
  81% urbanization     -> urbanization_3_0.pkl
  83% urbanization     -> urbanization_4_0.pkl
  87% urbanization     -> urbanization_5_0.pkl


In [3]:
# ── Export selected scenarios ─────────────────────────────────────────────────
# Maps each scenario key to the output CSV filename.
# Naming convention mirrors the legacy scripts for backward compatibility.
SCENARIO_FILENAME_MAP = {
    '38% urbanization': 'efrat_row_discharge_results_lowurbanization_level.csv',
    '87% urbanization': 'efrat_row_discharge_results_highurbanization_level.csv',
}
STATS_FILENAME_MAP = {
    '38% urbanization': 'efrat_event_stats_lowurbanization.csv',
    '87% urbanization': 'efrat_event_stats_highurbanization.csv',
}

results = {}
for scenario_key in EXPORT_SCENARIOS:
    out_csv   = SCENARIO_FILENAME_MAP.get(
        scenario_key,
        scenario_key.replace('%', 'pct').replace(' ', '_') + '_discharge.csv'
    )
    stats_csv = STATS_FILENAME_MAP.get(scenario_key)

    if not SIM_DIR.exists():
        logger.warning('Source directory not found: %s', SIM_DIR)
        logger.warning('Skipping %r (source data unavailable)', scenario_key)
        continue

    clean_df, event_stats = export_scenario(
        scenario_key=scenario_key,
        output_csv_filename=out_csv,
        sim_dir=SIM_DIR,
        output_dir=SCENARIOS_DIR,
        event_stats_filename=stats_csv,
    )
    results[scenario_key] = {'df': clean_df, 'stats': event_stats}
    logger.info('Exported %r -> %s', scenario_key, SCENARIOS_DIR / out_csv)

print(f'Exported {len(results)} scenario(s) to {SCENARIOS_DIR}')


22:01:24 INFO     urban_runoff.scenarios.export: Loading scenario '38% urbanization' from urbanization_1_0.pkl
22:01:24 INFO     urban_runoff.scenarios.export: Scenario '38% urbanization': raw shape (3444, 96) → filtered shape (3444, 34), saved to outputs\models\scenarios\efrat_row_discharge_results_lowurbanization_level.csv
22:01:24 INFO     urban_runoff.scenarios.export: Saved event stats to outputs\models\scenarios\efrat_event_stats_lowurbanization.csv, shape (41, 90)
22:01:24 INFO     scenarios_notebook: Exported '38% urbanization' -> outputs\models\scenarios\efrat_row_discharge_results_lowurbanization_level.csv
22:01:24 INFO     urban_runoff.scenarios.export: Loading scenario '87% urbanization' from urbanization_5_0.pkl
22:01:25 INFO     urban_runoff.scenarios.export: Scenario '87% urbanization': raw shape (3321, 96) → filtered shape (3321, 34), saved to outputs\models\scenarios\efrat_row_discharge_results_highurbanization_level.csv
22:01:25 INFO     urban_runoff.scenarios.export:

Exported 2 scenario(s) to outputs\models\scenarios


In [4]:
# ── Inspect exported data ─────────────────────────────────────────────────────
for scenario_key, data in results.items():
    df    = data['df']
    stats = data['stats']
    print(f'--- {scenario_key} ---')
    print(f'  Discharge shape  : {df.shape}')
    print(f'  Columns          : {list(df.columns[:5])} ...')
    if stats is not None:
        print(f'  Event stats shape: {stats.shape}')
        print(f'  Events found     : {list(stats.index)}')
    else:
        print('  Event stats      : not generated (no event column found)')
    print()


--- 38% urbanization ---
  Discharge shape  : (3444, 34)
  Columns          : [('event_num', '', ''), ('x', '', ''), ('y', '', ''), ('d', '', ''), ('historical', 'basin discharge', 'total_discharge')] ...
  Event stats shape: (41, 90)
  Events found     : ['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41']

--- 87% urbanization ---
  Discharge shape  : (3321, 34)
  Columns          : [('event_num', '', ''), ('x', '', ''), ('y', '', ''), ('d', '', ''), ('historical', 'basin discharge', 'total_discharge')] ...
  Event stats shape: (41, 90)
  Events found     : ['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40',

## Next Steps

The exported CSVs are now in `outputs/models/scenarios/` and are ready for the
WRF climate change analysis.

**Part 2 scope (future work):**
- Apply the calibrated SWMM model (Pareto optimum from Stage 2) to the WRF-driven
  rainfall scenarios.
- Compare historical vs. future urbanisation discharge profiles.
- Generate ensemble-uncertainty bands using the full Pareto ensemble from
  `outputs/pareto/final/pareto_ensemble_full.csv`.
